In [1]:
import os
import polars as pl

if not os.path.exists("datasets"):
    os.mkdir("datasets")

In [2]:
# Buscamos el nombre del archivo parquet del libro

book_file = [f for f in os.listdir("datasets") if "dataset_completo" in f and f.endswith(".parquet")][0]
path_book = os.path.join("datasets", book_file)

# Cargamos una vista previa
dataset_completo= pl.scan_parquet(path_book)

In [3]:
import polars as pl
import polars.selectors as cs

# 1. X_features: Todas las columnas MENOS las que empiezan por "label"
X_features = dataset_completo.select(~cs.starts_with("label"))

# 2. y_labels: SOLO las columnas que empiezan por "label"
y_labels = dataset_completo.select(cs.starts_with("label"))

In [4]:
def finalize_dataset(
    df_features: pl.DataFrame,
    df_labels: pl.DataFrame,
) -> pl.DataFrame:
    """Une features y labels y elimina los nulos generados por ventanas móviles.

    No aplicamos Z-score aquí: la normalización se hará más adelante,
    ajustando el StandardScaler SOLO con datos de entrenamiento de cada
    fold (y, para el modelo final, con todo el pool de CV), evitando así
    fuga de información (data leakage) hacia validación/test.
    """
    full_df = pl.concat([df_features, df_labels], how="horizontal")
    cleaned_df = full_df.drop_nulls()
    return cleaned_df


# Ensamblar y limpiar el dataset final
dataset_final = finalize_dataset(X_features, y_labels)

# Eliminamos columnas que no son features numéricas de entrada al modelo
dataset_final = dataset_final.drop([
    pl.col("mid_price_coordenada"),
    pl.col("^feat_.*$"),  # expresiones regulares nativas de Polars
])



In [5]:
import polars as pl
import pandas as pd
import numpy as np

TARGET_COL = "label_k_100"
CORR_THRESHOLD = 0.90  # Umbral para eliminar variables redundantes entre sí

# 1. Separar columnas
all_columns = dataset_final.collect_schema().names()
label_cols = [c for c in all_columns if c.startswith("label")]
candidate_feature_cols = [c for c in all_columns if c not in label_cols]

print(f"Calculando matriz de correlación para {len(candidate_feature_cols)} características...")

# 2. Construir expresiones para la matriz de correlación cruzada (todas contra todas)
# Nota: Si tienes miles de columnas, considera aplicar un .head(50000).collect() previo.
matrix_exprs = []
for c1 in candidate_feature_cols:
    for c2 in candidate_feature_cols:
        if c1 >= c2:  # Solo calculamos la mitad para ahorrar tiempo y procesamiento
            matrix_exprs.append(pl.corr(pl.col(c1), pl.col(c2)).alias(f"{c1}__VS__{c2}"))

# Calcular los valores en Polars
corr_results = dataset_final.select(matrix_exprs).collect()

# 3. Reconstruir la matriz simétrica en Pandas para usar sus herramientas de filtrado rápido
corr_matrix_pd = pd.DataFrame(index=candidate_feature_cols, columns=candidate_feature_cols, dtype=float)

for col in corr_results.columns:
    c1, c2 = col.split("__VS__")
    val = corr_results.get_column(col)[0]
    corr_matrix_pd.loc[c1, c2] = val
    corr_matrix_pd.loc[c2, c1] = val

# Llenar la diagonal con 1 por seguridad
np.fill_diagonal(corr_matrix_pd.values, 1.0)
corr_matrix_abs = corr_matrix_pd.abs()

# 4. Encontrar las características redundantes (con correlación > umbral entre ellas)
# Usamos un triángulo superior para no evaluar el mismo par dos veces
upper_tri = corr_matrix_abs.where(np.triu(np.ones(corr_matrix_abs.shape), k=1).astype(bool))

# Identificar qué columnas descartar
cols_to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > CORR_THRESHOLD)]

print(f"\n🔻 Columnas eliminadas por alta correlación ENTRE SÍ (> {CORR_THRESHOLD}) ({len(cols_to_drop)}):")
print(cols_to_drop)

# 5. Crear el nuevo dataset final sin las características redundantes
# Conservamos todas las labels intactas y las features que pasaron el filtro
final_features_to_keep = [c for c in candidate_feature_cols if c not in cols_to_drop]
dataset_filtrado = dataset_final.select(final_features_to_keep + label_cols)


Calculando matriz de correlación para 144 características...

🔻 Columnas eliminadas por alta correlación ENTRE SÍ (> 0.9) (74):
['v3', 'v5', 'v7', 'v9', 'v11', 'v13', 'v15', 'v17', 'v19', 'v21', 'v23', 'v25', 'v27', 'v29', 'v31', 'v33', 'v35', 'v37', 'v39', 'v47', 'v48', 'v49', 'v50', 'v57', 'v58', 'v59', 'v60', 'v67', 'v68', 'v69', 'v70', 'v72', 'v73', 'v74', 'v75', 'v76', 'v77', 'v78', 'v79', 'v80', 'v82', 'v83', 'v84', 'v85', 'v86', 'v87', 'v88', 'v89', 'v90', 'v92', 'v93', 'v94', 'v95', 'v96', 'v97', 'v98', 'v99', 'v100', 'v101', 'v102', 'v103', 'v104', 'v105', 'v106', 'v107', 'v108', 'v109', 'v110', 'v132', 'v137', 'v139', 'v140', 'v142', 'v143']


In [6]:
dataset_final=dataset_filtrado.drop([
    "label_k_10",
    "label_k_20",
    "label_k_30",
    "label_k_50"
])

In [10]:
dataset_final.collect().write_parquet('dataset_final.parquet')